In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

# Thêm các thư viện cho Model mới và Evaluation
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.model_selection import LeaveOneGroupOut

In [ ]:
# 1. Kết nối với Google Drive
drive.mount('/content/drive')

# 2. Đường dẫn file
# Tớ giữ nguyên đường dẫn cậu đã cung cấp
file_path = '/content/drive/MyDrive/3rd_Year/SPRING_2026/Signal/Final Project/master_dataset_aikido.csv'

# 3. Load dataset
try:
    df = pd.read_csv(file_path)
    print("✅ Đã load thành công dataset từ Google Drive!")
    print(f"Số lượng dòng dữ liệu: {df.shape[0]}")

    # Xem nhanh 5 dòng đầu để kiểm tra các cột
    display(df.head())

except FileNotFoundError:
    print("❌ Không tìm thấy file. Cậu hãy kiểm tra lại path hoặc tên file nhé.")

Mounted at /content/drive
✅ Đã load thành công dataset từ Google Drive!
Số lượng dòng dữ liệu: 39078


,Timestamp,AccX,AccY,AccZ,Heart_IR,Phase,Acc_Mag,Intensity_Label,Trial_ID,Property
0,0.009644,-556,42,1999,112276,0,2075.307447,3,INTENSE_01,CLEAN
1,0.049686,-618,48,1941,112252,0,2037.574293,3,INTENSE_01,CLEAN
2,0.089576,-588,51,1905,112378,0,1994.334475,3,INTENSE_01,CLEAN
3,0.128560,-595,53,1870,112421,0,1963.092968,3,INTENSE_01,CLEAN
4,0.168565,-575,68,1895,112533,0,1981.482778,3,INTENSE_01,CLEAN


In [ ]:
# 1. ĐỌC VÀ CHUẨN HÓA DATASET
punch_df = df[df['Phase'] == 1].copy()
trial_averages = punch_df.groupby('Trial_ID')['Acc_Mag'].mean()
intensive_trials = trial_averages[trial_averages > 6500].index.tolist()
punch_df['Binary_Intensity'] = punch_df['Trial_ID'].apply(lambda x: 1 if x in intensive_trials else 0)

# 2. TRÍCH XUẤT ĐẶC TRƯNG CỬA SỔ
window_size = 60
stride = 10
X_features, y_labels, groups = [], [], []

for trial, group in punch_df.groupby('Trial_ID'):
    acc_mag_signals = group['Acc_Mag'].values
    binary_labels = group['Binary_Intensity'].values
    for i in range(0, len(acc_mag_signals) - window_size + 1, stride):
        window_data = acc_mag_signals[i : i + window_size]
        acc_std = np.std(window_data, ddof=0)
        if acc_std < 1500.0: continue # Bộ lọc khoảng lặng

        peak_max = np.max(window_data)
        mean_mag = np.mean(window_data)
        peak_relative = peak_max / mean_mag if mean_mag > 0 else 0
        window_label = 1 if np.mean(binary_labels[i : i + window_size]) >= 0.5 else 0

        X_features.append([peak_max, acc_std, peak_relative])
        y_labels.append(window_label)
        groups.append(trial)

X_all, y_all, groups = np.array(X_features), np.array(y_labels), np.array(groups)

# 3. CHẠY KIỂM THỬ CHÉO XOAY VÒNG LEAVE-ONE-GROUP-OUT (LOGO-CV)
logo = LeaveOneGroupOut()
train_scores, test_scores = [], []

print("--- TIẾN TRÌNH KIỂM THỬ CHÉO XOAY VÒNG (LOGO-CV) ---")
for train_idx, test_idx in logo.split(X_all, y_all, groups):
    X_train, X_test = X_all[train_idx], X_all[test_idx]
    y_train, y_test = y_all[train_idx], y_all[test_idx]
    test_person = groups[test_idx][0]

    clf_fold = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=42)
    clf_fold.fit(X_train, y_train)

    train_scores.append(accuracy_score(y_train, clf_fold.predict(X_train)))
    test_scores.append(accuracy_score(y_test, clf_fold.predict(X_test)))
    print(f"• Thử nghiệm chặn người dùng '{test_person}': Test Accuracy = {test_scores[-1]*100:.2f}%")

print("\n=======================================================")
print("     KẾT QUẢ KIỂM ĐỊNH TỔNG THỂ ĐẠT CHUẨN KHOA HỌC     ")
print("=======================================================")
print(f"👉 Trung bình Accuracy tập Train: {np.mean(train_scores)*100:.2f}%")
print(f"👉 Trung bình Accuracy tập Test : {np.mean(test_scores)*100:.2f}%")
print(f"👉 Khoảng chênh lệch thực tế    : {abs(np.mean(train_scores) - np.mean(test_scores))*100:.2f}%")
print("=======================================================")

# 4. TRAIN MÔ HÌNH CUỐI CÙNG TRÊN TOÀN BỘ DATASET ĐỂ XUẤT FILE C++
final_clf = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=42)
final_clf.fit(X_all, y_all)

# (Đoạn code generate_cpp_code giữ nguyên như cũ để xuất file classifier.h)

--- TIẾN TRÌNH KIỂM THỬ CHÉO XOAY VÒNG (LOGO-CV) ---
• Thử nghiệm chặn người dùng 'INTENSE_01': Test Accuracy = 79.55%
• Thử nghiệm chặn người dùng 'INTENSE_03': Test Accuracy = 100.00%
• Thử nghiệm chặn người dùng 'INTENSE_04': Test Accuracy = 90.65%
• Thử nghiệm chặn người dùng 'INTENSE_05': Test Accuracy = 98.13%
• Thử nghiệm chặn người dùng 'INTENSE_06': Test Accuracy = 100.00%
• Thử nghiệm chặn người dùng 'INTENSE_07': Test Accuracy = 94.39%
• Thử nghiệm chặn người dùng 'LIGHT_01': Test Accuracy = 100.00%
• Thử nghiệm chặn người dùng 'LIGHT_02': Test Accuracy = 100.00%
• Thử nghiệm chặn người dùng 'MEDIUM_01': Test Accuracy = 27.59%
• Thử nghiệm chặn người dùng 'MEDIUM_02': Test Accuracy = 100.00%
• Thử nghiệm chặn người dùng 'MEDIUM_03': Test Accuracy = 98.57%

     KẾT QUẢ KIỂM ĐỊNH TỔNG THỂ ĐẠT CHUẨN KHOA HỌC     
👉 Trung bình Accuracy tập Train: 93.82%
👉 Trung bình Accuracy tập Test : 89.90%
👉 Khoảng chênh lệch thực tế    : 3.93%


DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=42)

In [ ]:
# =====================================================================
# 5. HÀM TỰ ĐỘNG DỊCH CÂY QUYẾT ĐỊNH THÀNH MÃ C++ (CLASSIFIER.H)
# =====================================================================
def generate_cpp_code(model, feature_names):
    tree = model.tree_
    left = tree.children_left
    right = tree.children_right
    threshold = tree.threshold
    features = tree.feature
    value = tree.value

    def recurse(node, depth):
        indent = "    " * depth
        if left[node] == -1:
            return f"{indent}return {np.argmax(value[node][0])}; // Class {np.argmax(value[node][0])}\n"
        else:
            name = feature_names[features[node]]
            th = threshold[node]
            cpp_str = f"{indent}if ({name} <= {th:.2f}) {{\n"
            cpp_str += recurse(left[node], depth + 1)
            cpp_str += f"{indent}}} else {{\n"
            cpp_str += recurse(right[node], depth + 1)
            cpp_str += f"{indent}}}\n"
            return cpp_str

    code = "#ifndef CLASSIFIER_H\n#define CLASSIFIER_H\n\n"
    code += "int classifySignal(float peak_max, float acc_std, float peak_relative) {\n"
    code += recurse(0, 1)
    code += "}\n\n#endif"
    return code

print("\n[Ý 5] Đang tự động dịch mô hình học máy biên sang mã C++...")
cpp_classifier = generate_cpp_code(final_clf, ['peak_max', 'acc_std', 'peak_relative'])

# Ghi file cứng lên đĩa đệm của Colab
with open("classifier.h", "w", encoding="utf-8") as f:
    f.write(cpp_classifier)
print("✅ [THÀNH CÔNG]: File 'classifier.h' đã xuất hiện tại mục Files bên trái!")


[Ý 5] Đang tự động dịch mô hình học máy biên sang mã C++...
✅ [THÀNH CÔNG]: File 'classifier.h' đã xuất hiện tại mục Files bên trái!
